In [ ]:
import sys
import os

# Go to the cloned repo folder
repo_path = '/content/NLP-sequence-classification'

# Add to Python path
sys.path.append(repo_path)

# Change working directory to repo
os.chdir(repo_path)

In [ ]:
from src.data_loader import get_BESSTIE_splits
import pandas as pd
from src.lr_feature_extraction import tfidf_features, load_tfidf_features
from models.svm_tfidf import MultiOutputSVM, SeparateSVM

In [ ]:
# loading and getting the splits from the dataset
df_all, df_train, df_validation, df_test = get_BESSTIE_splits()

In [ ]:
# Extract TFIDF Features
X_train, X_validation, X_test, vectorizer = tfidf_features(
    df_train, df_validation, df_test,
    text_column='text',
    max_features=15000,
    save_path="./models/tfidf"
)

## Multi-Output SVM
Single model predicting both Sarcasm and Sentiment simultaneously

In [ ]:
# train the multi-output model
multi_svm = MultiOutputSVM()
multi_svm.train_MultiOutputSVM(X_train, df_train)

In [ ]:
# Evaluate the multi-output model
multi_svm_results = multi_svm.MultiOutputSVM_evaluation(X_validation, df_validation)

print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {multi_svm_results['Sarcasm']['Accuracy']:.4f}")
print(f"   Precision: {multi_svm_results['Sarcasm']['Precision']:.4f}")
print(f"   Recall:    {multi_svm_results['Sarcasm']['Recall']:.4f}")
print(f"   F1-Score:  {multi_svm_results['Sarcasm']['F1']:.4f}")
print(f"   F1-Macro:  {multi_svm_results['Sarcasm']['F1_Macro']:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {multi_svm_results['Sentiment']['Accuracy']:.4f}")
print(f"   Precision: {multi_svm_results['Sentiment']['Precision']:.4f}")
print(f"   Recall:    {multi_svm_results['Sentiment']['Recall']:.4f}")
print(f"   F1-Score:  {multi_svm_results['Sentiment']['F1']:.4f}")
print(f"   F1-Macro:  {multi_svm_results['Sentiment']['F1_Macro']:.4f}")

In [ ]:
# Save multi-output model
multi_svm.save_MultiOutputSVM_model("./models/multi_output_svm.pkl")

## Separate SVM Models — BESSTIE Paper Approach
One model for Sarcasm, one model for Sentiment

In [ ]:
# train separate models
separate_svm = SeparateSVM()
separate_svm.train_SeparateSVM(X_train, df_train)

In [ ]:
# Evaluate separate models
separate_svm_results = separate_svm.SeparateSVM_evaluation(X_validation, df_validation)

print("\n📌 SARCASM DETECTION:")
print(f"   Accuracy:  {separate_svm_results['Sarcasm']['Accuracy']:.4f}")
print(f"   Precision: {separate_svm_results['Sarcasm']['Precision']:.4f}")
print(f"   Recall:    {separate_svm_results['Sarcasm']['Recall']:.4f}")
print(f"   F1-Score:  {separate_svm_results['Sarcasm']['F1']:.4f}")
print(f"   F1-Macro:  {separate_svm_results['Sarcasm']['F1_Macro']:.4f}")

print("\n📌 SENTIMENT ANALYSIS:")
print(f"   Accuracy:  {separate_svm_results['Sentiment']['Accuracy']:.4f}")
print(f"   Precision: {separate_svm_results['Sentiment']['Precision']:.4f}")
print(f"   Recall:    {separate_svm_results['Sentiment']['Recall']:.4f}")
print(f"   F1-Score:  {separate_svm_results['Sentiment']['F1']:.4f}")
print(f"   F1-Macro:  {separate_svm_results['Sentiment']['F1_Macro']:.4f}")

In [ ]:
# Save separate models
separate_svm.save_SeparateSVM_models("./models")

## FINAL TEST EVALUATION

In [ ]:
# FINAL TEST EVALUATION — Separate SVM (BESSTIE paper approach)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

test_predictions = separate_svm.prediction_SeparateSVM(X_test)

sarcasm_true_labels = df_test['Sarcasm'].astype(int).values
sentiment_true_labels = df_test['Sentiment'].astype(int).values

sarcasm_f1        = f1_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_f1_macro  = f1_score(sarcasm_true_labels, test_predictions['Sarcasm'], average='macro')
sarcasm_precision = precision_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_recall    = recall_score(sarcasm_true_labels, test_predictions['Sarcasm'])
sarcasm_accuracy  = accuracy_score(sarcasm_true_labels, test_predictions['Sarcasm'])

sentiment_f1        = f1_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_f1_macro  = f1_score(sentiment_true_labels, test_predictions['Sentiment'], average='macro')
sentiment_precision = precision_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_recall    = recall_score(sentiment_true_labels, test_predictions['Sentiment'])
sentiment_accuracy  = accuracy_score(sentiment_true_labels, test_predictions['Sentiment'])

print("\n📌 SARCASM DETECTION (Test):")
print(f"   Accuracy:  {sarcasm_accuracy:.4f}")
print(f"   Precision: {sarcasm_precision:.4f}")
print(f"   Recall:    {sarcasm_recall:.4f}")
print(f"   F1-Score:  {sarcasm_f1:.4f}")
print(f"   F1-Macro:  {sarcasm_f1_macro:.4f}")

print("\n📌 SENTIMENT ANALYSIS (Test):")
print(f"   Accuracy:  {sentiment_accuracy:.4f}")
print(f"   Precision: {sentiment_precision:.4f}")
print(f"   Recall:    {sentiment_recall:.4f}")
print(f"   F1-Score:  {sentiment_f1:.4f}")
print(f"   F1-Macro:  {sentiment_f1_macro:.4f}")